In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Kernel Fisher Discriminant Analysis (KFDA) + Inverse Class-Weighted Linear SVM (`models/train_distant_analysis.ipynb`)

Loads the full emergency cohort from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI) and executes **Kernel Fisher Discriminant Analysis (KFDA)** with **Nystroem Non-Linear RBF Kernel Approximation** across the 8 core arrival triage features, paired with **Inverse Class Frequency Weighting** for **`ESI 1 (Immediate Resuscitation)`** in the **Linear Support Vector Machine (Linear SVM)** classifier.

```mermaid
flowchart TD
    Raw["Raw Arrival Features X in R^8 (558,029 Visits)"] --> Imputer["SimpleImputer(strategy='median') & StandardScaler()"]
    Imputer --> Nystroem["Nystroem Non-Linear RBF Map phi(X) in R^600"]
    Nystroem --> KFDA["Kernel Fisher Discriminant Analysis (KFDA)"]
    KFDA --> Manifold["2D KFDA Manifold Coordinates [z1: KFDA Axis, z2: Orthogonal Kernel PC]"]
    Manifold --> LinSVM["Linear SVM with Inverse Class Weighting (w_ESI1 ~ 105x Penalty)"]
    LinSVM --> Eval["Holdout Test Evaluation (83k Visits) & Before/After Scatter Visualizations"]
```

### 🎯 Target Formulation
- **`Class 1: ESI 1 (Immediate Resuscitation)`** ($y=1$): Patients requiring immediate life-saving intervention ($5,271$ visits, $\sim 0.94\%$).
- **`Class 0: NOT ESI 1 (ESI 2–5)`** ($y=0$): Emergent, urgent, and non-urgent visits ($552,758$ visits, $\sim 99.06\%$).

### 🩺 8 Core Arrival Triage Features
1. `age`
2. `cc_breathingdifficulty`
3. `gender` (0 = Female, 1 = Male)
4. `triage_vital_hr` (Heart Rate)
5. `triage_vital_sbp` (Systolic Blood Pressure)
6. `triage_vital_dbp` (Diastolic Blood Pressure)
7. `triage_vital_rr` (Respiratory Rate)
8. `triage_vital_o2` (Oxygen Saturation - SpO2)

### ⚖️ Inverse Class Weighting Formula
To prevent the $99.06\%$ majority class from overwhelming the classifier, sample weights are inversely proportional to class frequencies:

$$w_k = \frac{N}{2 \times N_k} \implies w_0 \approx 0.505, \quad w_1 \approx 52.93 \quad \left(\frac{w_1}{w_0} \approx 104.87\right)$$

In the Linear SVM objective function:

$$\min_{w, b} \frac{1}{2} ||w||^2 + C \sum_{i=1}^N w_{y_i} \max(0, 1 - y_i (w^T z_i + b))$$

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA (retaining all ~558k observations)
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 core triage features + ESI
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c(
  "age", "cc_breathingdifficulty", "gender",
  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"
)

# Export matrices to Python (NAs preserved for SimpleImputer)
raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported Full Dataset to Python: %d rows, %d feature columns\n",
            nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve Data from R, Partition & Apply Preprocessing
# ---------------------------------------------------------------------------
import os, json, pickle, warnings
from rpy2.robjects import r
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight
from sklearn.kernel_approximation import Nystroem
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, precision_score,
                             f1_score, fbeta_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report, roc_curve, precision_recall_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender',
    'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
    'triage_vital_rr', 'triage_vital_o2'
]

# Binary Target: 1 = ESI 1 (Immediate Resuscitation), 0 = NOT ESI 1 (ESI 2-5)
y_all = np.where(esi_all == 1, 1, 0)
LABELS = ['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)']

print("=========================================================================")
print("     5v_cleandf COHORT FOR INVERSE CLASS-WEIGHTED KFDA + SVM")
print("=========================================================================")
print(f"Total Valid ESI Visits: {len(y_all):,}")
print(f"  * Class 0 [NOT ESI 1 (ESI 2-5)]: {np.sum(y_all == 0):,} ({np.mean(y_all == 0)*100:.2f}%)")
print(f"  * Class 1 [ESI 1 (Resuscitation)]: {np.sum(y_all == 1):,} ({np.mean(y_all == 1)*100:.2f}%)")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print("=========================================================================\n")

# Stratified 3-way split: 70% Train, 15% Validation, 15% Holdout Test
itr, itmp = train_test_split(np.arange(len(y_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

raw_tr  = raw_mat_all[itr]
raw_val = raw_mat_all[iva]
raw_te  = raw_mat_all[ite]

y_train = y_all[itr]
y_val   = y_all[iva]
y_test  = y_all[ite]

# Compute exact inverse class frequency weights on Training partition
classes = np.array([0, 1])
computed_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {0: float(computed_weights[0]), 1: float(computed_weights[1])}

print(f"✓ Computed Inverse Class Weights:")
print(f"  * Weight for NOT ESI 1 (Class 0): {class_weight_dict[0]:.4f}")
print(f"  * Weight for ESI 1 (Class 1)    : {class_weight_dict[1]:.4f} (Ratio: {class_weight_dict[1]/class_weight_dict[0]:.2f}x penalty)")

# Fit SimpleImputer & StandardScaler strictly on Training partition
print("\nFitting SimpleImputer(strategy='median') & StandardScaler on Training set...")
imputer = SimpleImputer(strategy='median')
X_tr_imp  = imputer.fit_transform(raw_tr)
X_val_imp = imputer.transform(raw_val)
X_te_imp  = imputer.transform(raw_te)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_tr_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_te_imp)

print(f"✓ Partition Shapes: Train={X_train_scaled.shape}, Val={X_val_scaled.shape}, Test={X_test_scaled.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Fit Non-Linear Nystroem Kernel Map & KFDA on Scaled Features
# ---------------------------------------------------------------------------
print("Building Non-Linear RBF Kernel Feature Map via Nystroem Approximation...")

# Tune RBF gamma for maximal Fisher Separability J(w)
gamma_candidates = [0.05, 0.10, 0.15, 0.20, 0.30]
best_gamma = 0.15
best_fisher_ratio = -1.0

# Stratified subsample for tuning
np.random.seed(42)
tune_esi1_idx = np.where(y_train == 1)[0]
tune_not_idx  = np.where(y_train == 0)[0]
tune_idx = np.concatenate([
    tune_esi1_idx,
    np.random.choice(tune_not_idx, min(35000, len(tune_not_idx)), replace=False)
])
X_tune = X_train_scaled[tune_idx]
y_tune = y_train[tune_idx]

print("Scanning RBF Gamma candidates for maximal Fisher Separability J(w):")
for g in gamma_candidates:
    nyst_tune = Nystroem(kernel='rbf', gamma=g, n_components=300, random_state=42, n_jobs=-1)
    Phi_tune = nyst_tune.fit_transform(X_tune)
    lda_tune = LinearDiscriminantAnalysis(n_components=1)
    z_tune = lda_tune.fit_transform(Phi_tune, y_tune).ravel()
    
    z_pos = z_tune[y_tune == 1]
    z_neg = z_tune[y_tune == 0]
    j_score = ((np.mean(z_pos) - np.mean(z_neg))**2) / (np.var(z_pos) + np.var(z_neg) + 1e-10)
    auc_s = roc_auc_score(y_tune, z_tune)
    if auc_s < 0.5: auc_s = 1.0 - auc_s
    print(f"  * Gamma={g:.2f} -> Fisher Ratio J(w) = {j_score:.4f} | KFDA AUC = {auc_s:.4f}")
    if j_score > best_fisher_ratio:
        best_fisher_ratio = j_score
        best_gamma = g

print(f"\n✓ Selected Optimal RBF Gamma: {best_gamma}")

# Fit Final Nystroem Feature Map (m=600 landmark components)
nystroem = Nystroem(kernel='rbf', gamma=best_gamma, n_components=600, random_state=42, n_jobs=-1)
print("Transforming full training dataset: R^8 -> R^600...")
Phi_train = nystroem.fit_transform(X_train_scaled)
Phi_val   = nystroem.transform(X_val_scaled)
Phi_test  = nystroem.transform(X_test_scaled)

# Solve Kernel Fisher Discriminant Analysis (KFDA)
print("Solving Kernel Fisher Discriminant Analysis on Phi...")
kfda = LinearDiscriminantAnalysis(n_components=1)
kfda.fit(Phi_train, y_train)

# Compute 1D Discriminant Scores
z1_tr  = kfda.transform(Phi_train).ravel()
z1_val = kfda.transform(Phi_val).ravel()
z1_te  = kfda.transform(Phi_test).ravel()

# Extract Orthogonal Kernel Principal Component to construct full 2D KFDA Manifold
print("Extracting Orthogonal Principal Kernel Component for 2D Manifold Space...")
w_kfd = kfda.coef_  # (1, 600)
w_norm = w_kfd / np.linalg.norm(w_kfd)
Phi_ortho_tr  = Phi_train - np.dot(Phi_train, w_norm.T) * w_norm
Phi_ortho_val = Phi_val - np.dot(Phi_val, w_norm.T) * w_norm
Phi_ortho_te  = Phi_test - np.dot(Phi_test, w_norm.T) * w_norm

pca_ortho = PCA(n_components=1, random_state=42)
z2_tr  = pca_ortho.fit_transform(Phi_ortho_tr).ravel()
z2_val = pca_ortho.transform(Phi_ortho_val).ravel()
z2_te  = pca_ortho.transform(Phi_ortho_te).ravel()

# Assemble 2D KFDA Manifold Matrices: Z = [z1 (KFDA Axis), z2 (Ortho Kernel PC)]
Z_train = np.column_stack([z1_tr, z2_tr])
Z_val   = np.column_stack([z1_val, z2_val])
Z_test  = np.column_stack([z1_te, z2_te])

# 2D Linear PCA of Raw 8 Features (for Before-Projection Baseline)
pca_raw = PCA(n_components=2, random_state=42)
X_raw_pca_te = pca_raw.fit_transform(X_test_scaled)

print(f"✓ 2D KFDA Manifold Assembled: Train={Z_train.shape}, Val={Z_val.shape}, Test={Z_test.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Train Linear SVM with Inverse Class Frequency Weighting on KFDA Manifold
# ---------------------------------------------------------------------------
print(f"Training Linear SVM with Inverse Class Weighting: {class_weight_dict}...")

# Linear Support Vector Machine with Inverse Class Weighting on ESI 1
svm_base = LinearSVC(
    C=1.0,
    class_weight=class_weight_dict,  # Inversely weights ESI 1 by ~105x
    random_state=42,
    max_iter=15000,
    dual='auto'
)
svm_base.fit(Z_train, y_train)

# Calibrate Linear SVM on Validation Split (Sigmoid / Platt Scaling)
print("Calibrating Linear SVM on Validation Split (Platt/Sigmoid Scaling)...")
svm_calibrated = CalibratedClassifierCV(svm_base, method='sigmoid', cv='prefit')
svm_calibrated.fit(Z_val, y_val)

# Extract Linear SVM Hyperplane parameters
w_svm = svm_base.coef_[0]
b_svm = svm_base.intercept_[0]
print(f"\n✓ Inverse-Weighted Linear SVM Successfully Trained on KFDA Space!")
print(f"  * Hyperplane Equation : {w_svm[0]:.4f} * z1 + {w_svm[1]:.4f} * z2 + ({b_svm:.4f}) = 0")
print(f"  * KFDA Axis Weight    : {w_svm[0]:.4f} (Dominant Discriminant Direction)")
print(f"  * Orthogonal PC Weight: {w_svm[1]:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Holdout Test Set Evaluation & Inverse Class Weight Sensitivity Sweep
# ---------------------------------------------------------------------------
# 1. Evaluate Primary Inverse-Weighted Model on Holdout Test Set
p_test_esi1 = svm_calibrated.predict_proba(Z_test)[:, 1]
pred_test   = (p_test_esi1 >= 0.50).astype(int)
dec_func_te = svm_base.decision_function(Z_test)

acc      = accuracy_score(y_test, pred_test)
bal_acc  = balanced_accuracy_score(y_test, pred_test)
rec_esi1 = recall_score(y_test == 1, pred_test == 1, zero_division=0)
spec_not = recall_score(y_test == 0, pred_test == 0, zero_division=0)
prec_esi1= precision_score(y_test == 1, pred_test == 1, zero_division=0)
f1_esi1  = f1_score(y_test == 1, pred_test == 1, zero_division=0)
f05_esi1 = fbeta_score(y_test == 1, pred_test == 1, beta=0.5, zero_division=0)
auc_val  = roc_auc_score(y_test, p_test_esi1)
pr_auc   = average_precision_score(y_test, p_test_esi1)

# 2. Sensitivity Sweep: Impact of ESI 1 Class Weight on Precision vs Recall Tradeoff
weight_multipliers = [1.0, 10.0, 25.0, 50.0, 105.0, 200.0]
sweep_rows = []

for mult in weight_multipliers:
    cw_test = {0: 1.0, 1: float(mult)}
    svm_temp = LinearSVC(C=1.0, class_weight=cw_test, random_state=42, max_iter=10000, dual='auto')
    svm_temp.fit(Z_train, y_train)
    calib_temp = CalibratedClassifierCV(svm_temp, method='sigmoid', cv='prefit')
    calib_temp.fit(Z_val, y_val)
    
    p_tmp = calib_temp.predict_proba(Z_test)[:, 1]
    pred_tmp = (p_tmp >= 0.50).astype(int)
    
    rec_tmp  = recall_score(y_test == 1, pred_tmp == 1, zero_division=0)
    spec_tmp = recall_score(y_test == 0, pred_tmp == 0, zero_division=0)
    prec_tmp = precision_score(y_test == 1, pred_tmp == 1, zero_division=0)
    f1_tmp   = f1_score(y_test == 1, pred_tmp == 1, zero_division=0)
    bacc_tmp = balanced_accuracy_score(y_test, pred_tmp)
    
    sweep_rows.append({
        'ESI1_Weight_Multiplier': f"{mult:.0f}x",
        'ESI1_Sensitivity (Recall)': f"{rec_tmp*100:.2f}%",
        'Specificity': f"{spec_tmp*100:.2f}%",
        'ESI1_Precision': f"{prec_tmp*100:.2f}%",
        'Balanced_Accuracy': f"{bacc_tmp*100:.2f}%",
        'F1_Score': round(f1_tmp, 4)
    })

print("=========================================================================================================")
print("    INVERSE CLASS WEIGHT SENSITIVITY SWEEP: PRECISION vs SENSITIVITY TRADEOFF")
print("=========================================================================================================")
sweep_df = pd.DataFrame(sweep_rows)
print(sweep_df.to_string(index=False))
print("=========================================================================================================\n")

report_data = [{
    'Model': 'KFDA + Inverse-Weighted Linear SVM',
    'ESI1_Weight': f"{class_weight_dict[1]/class_weight_dict[0]:.1f}x (Inverse Frequency)",
    'Accuracy': round(acc, 4),
    'Balanced_Accuracy': round(bal_acc, 4),
    'ESI1_Sensitivity (Recall)': round(rec_esi1, 4),
    'Specificity (NOT ESI 1 Recall)': round(spec_not, 4),
    'ESI1_Precision': round(prec_esi1, 4),
    'ESI1_F1': round(f1_esi1, 4),
    'ESI1_F0.5': round(f05_esi1, 4),
    'ROC_AUC': round(auc_val, 4),
    'PR_AUC': round(pr_auc, 4)
}]

report_df = pd.DataFrame(report_data)
print("Primary Model Holdout Performance:")
print(report_df.to_string(index=False))
print("\nDetailed Classification Report:")
print(classification_report(y_test, pred_test, target_names=LABELS, digits=4))

reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'kfda_inverse_weighted_svm_report.csv')
report_df.to_csv(report_file, index=False)
print(f"Metrics report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: Confusion Matrix Heatmap for Inverse-Weighted KFDA + Linear SVM
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'distant_analysis'), exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(7.5, 6))
annot = np.empty_like(cm, dtype=object)
for i in range(2):
    for j in range(2):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.2f}%)"

sns.heatmap(
    cm_norm,
    annot=annot,
    fmt='',
    cmap='Blues',
    cbar=True,
    ax=ax,
    vmin=0,
    vmax=1,
    xticklabels=['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)'],
    yticklabels=['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)']
)

ax.set_title(
    f'Inverse-Weighted KFDA + Linear SVM: Confusion Matrix (Holdout Test)\n'
    f'Accuracy: {acc*100:.2f}% | ESI 1 Sensitivity: {rec_esi1*100:.2f}% | Specificity: {spec_not*100:.2f}%',
    fontsize=11.5,
    fontweight='bold',
    pad=12
)
ax.set_xlabel('Predicted Acuity Label', fontsize=11, fontweight='bold')
ax.set_ylabel('True Acuity Label', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, 'distant_analysis', 'kfda_linear_svm_confusion_matrix.png')
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'kfda_linear_svm_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion Matrix saved to: {cm_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: BEFORE vs AFTER Scatter Plots & Inverse-Weighted Linear SVM Boundary
# ---------------------------------------------------------------------------
np.random.seed(42)
test_esi1_idx = np.where(y_test == 1)[0]
test_not_idx  = np.where(y_test == 0)[0]
test_not_sample = np.random.choice(test_not_idx, min(2500, len(test_not_idx)), replace=False)

fig, axes = plt.subplots(2, 2, figsize=(17, 14))

# Panel 1: BEFORE - Heart Rate vs Systolic Blood Pressure
hr_idx  = FEATURES.index('triage_vital_hr')
sbp_idx = FEATURES.index('triage_vital_sbp')

axes[0, 0].scatter(X_te_imp[test_not_sample, hr_idx], X_te_imp[test_not_sample, sbp_idx],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (Sample)')
axes[0, 0].scatter(X_te_imp[test_esi1_idx, hr_idx], X_te_imp[test_esi1_idx, sbp_idx],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='ESI 1 (Resuscitation)')
axes[0, 0].set_title('BEFORE: Raw Vital Features (Heart Rate vs Systolic BP)\n[Heavy Class Overlap & Non-Linear Entanglement]', fontsize=11.5, fontweight='bold', pad=10)
axes[0, 0].set_xlabel('Heart Rate (bpm)', fontsize=10.5, fontweight='bold')
axes[0, 0].set_ylabel('Systolic BP (mmHg)', fontsize=10.5, fontweight='bold')
axes[0, 0].set_xlim(30, 200)
axes[0, 0].set_ylim(50, 240)
axes[0, 0].grid(True, linestyle=':', alpha=0.4)
axes[0, 0].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 2: BEFORE - Respiratory Rate vs SpO2
rr_idx = FEATURES.index('triage_vital_rr')
o2_idx = FEATURES.index('triage_vital_o2')

axes[0, 1].scatter(X_te_imp[test_not_sample, rr_idx], X_te_imp[test_not_sample, o2_idx],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (Sample)')
axes[0, 1].scatter(X_te_imp[test_esi1_idx, rr_idx], X_te_imp[test_esi1_idx, o2_idx],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='ESI 1 (Resuscitation)')
axes[0, 1].set_title('BEFORE: Raw Vital Features (Respiratory Rate vs SpO2)\n[Severe Overlap in Clinical Oxygenation Ranges]', fontsize=11.5, fontweight='bold', pad=10)
axes[0, 1].set_xlabel('Respiratory Rate (bpm)', fontsize=10.5, fontweight='bold')
axes[0, 1].set_ylabel('Oxygen Saturation SpO2 (%)', fontsize=10.5, fontweight='bold')
axes[0, 1].set_xlim(6, 45)
axes[0, 1].set_ylim(70, 100)
axes[0, 1].grid(True, linestyle=':', alpha=0.4)
axes[0, 1].legend(loc='lower left', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 3: BEFORE - 2D Linear PCA of Raw 8 Features
axes[1, 0].scatter(X_raw_pca_te[test_not_sample, 0], X_raw_pca_te[test_not_sample, 1],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (Sample)')
axes[1, 0].scatter(X_raw_pca_te[test_esi1_idx, 0], X_raw_pca_te[test_esi1_idx, 1],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='ESI 1 (Resuscitation)')
axes[1, 0].set_title('BEFORE: 2D Linear PCA Baseline (Original 8-D Space)\n[Linear Unsupervised Projection Fails to Separate Classes]', fontsize=11.5, fontweight='bold', pad=10)
axes[1, 0].set_xlabel('Linear Principal Component 1', fontsize=10.5, fontweight='bold')
axes[1, 0].set_ylabel('Linear Principal Component 2', fontsize=10.5, fontweight='bold')
axes[1, 0].grid(True, linestyle=':', alpha=0.4)
axes[1, 0].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 4: AFTER - 2D KFDA Manifold with Inverse-Weighted Linear SVM Boundary
z1_min, z1_max = Z_test[:, 0].min() - 0.5, Z_test[:, 0].max() + 0.5
z2_min, z2_max = Z_test[:, 1].min() - 0.5, Z_test[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(z1_min, z1_max, 200), np.linspace(z2_min, z2_max, 200))
grid_z = np.c_[xx.ravel(), yy.ravel()]

# Linear SVM probability contours
probs_grid = svm_calibrated.predict_proba(grid_z)[:, 1].reshape(xx.shape)
cf = axes[1, 1].contourf(xx, yy, probs_grid, levels=np.linspace(0, 1, 11), cmap='RdYlBu_r', alpha=0.60, vmin=0, vmax=1)

# Linear SVM Decision Boundary (Solid Black Line at P=0.50)
cs_decision = axes[1, 1].contour(xx, yy, probs_grid, levels=[0.50], colors='black', linewidths=2.8, linestyles='-')
axes[1, 1].clabel(cs_decision, inline=True, fontsize=10, fmt='SVM Separator (P=0.50)')

# Overlay Test Scatter Points
axes[1, 1].scatter(Z_test[test_not_sample, 0], Z_test[test_not_sample, 1],
                   c='#1f77b4', alpha=0.35, s=18, label='NOT ESI 1 (Sample)')
axes[1, 1].scatter(Z_test[test_esi1_idx, 0], Z_test[test_esi1_idx, 1],
                   c='#d62728', alpha=0.85, s=40, edgecolors='black', linewidth=0.6, label='ESI 1 (Resuscitation)')

axes[1, 1].set_title(f'AFTER: KFDA Manifold + Inverse-Weighted Linear SVM Boundary\n[Separating Hyperplane: {w_svm[0]:.2f}*z1 + {w_svm[1]:.2f}*z2 + ({b_svm:.2f}) = 0 | Weight={class_weight_dict[1]/class_weight_dict[0]:.1f}x]', fontsize=11.5, fontweight='bold', pad=10)
axes[1, 1].set_xlabel('Component 1: Kernel Fisher Discriminant Axis (z1)', fontsize=10.5, fontweight='bold')
axes[1, 1].set_ylabel('Component 2: Orthogonal Kernel PC (z2)', fontsize=10.5, fontweight='bold')
axes[1, 1].set_xlim(z1_min, z1_max)
axes[1, 1].set_ylim(z2_min, z2_max)
axes[1, 1].grid(True, linestyle=':', alpha=0.4)
axes[1, 1].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)
cbar = plt.colorbar(cf, ax=axes[1, 1], fraction=0.046, pad=0.04)
cbar.set_label('Linear SVM Predicted P(ESI 1)', fontsize=9.5, fontweight='bold')

plt.suptitle('Non-Linear Distance Analysis: Nystroem KFDA Feature Projection & Inverse-Weighted Linear SVM\nBEFORE vs AFTER Comparison on Holdout Test Set', fontsize=14.5, fontweight='bold', y=0.995)
plt.tight_layout()

scatter_file = os.path.join(plots_dir, 'distant_analysis', 'kfda_linear_svm_before_after_scatter.png')
plt.savefig(scatter_file, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'kfda_linear_svm_before_after_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Before/After SVM scatter plot saved to: {scatter_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Linear SVM Decision Function Margin & ROC / PR Curves
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: SVM Decision Function Distribution (Distance to Separating Hyperplane)
sns.kdeplot(dec_func_te[y_test == 0], ax=axes[0], color='#1f77b4', fill=True, alpha=0.35, linewidth=2.0, label='NOT ESI 1 (ESI 2-5)')
sns.kdeplot(dec_func_te[y_test == 1], ax=axes[0], color='#d62728', fill=True, alpha=0.35, linewidth=2.0, label='ESI 1 (Resuscitation)')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=2.0, label='Linear SVM Decision Boundary (Margin = 0)')
axes[0].set_title('Inverse-Weighted Linear SVM Decision Margin on KFDA Space\n[Distance from Maximum-Margin Separator w*z + b]', fontsize=12, fontweight='bold', pad=10)
axes[0].set_xlabel('Signed Distance to Separating Hyperplane (w*z + b)', fontsize=10.5, fontweight='bold')
axes[0].set_ylabel('Probability Density', fontsize=10.5, fontweight='bold')
axes[0].grid(True, linestyle=':', alpha=0.4)
axes[0].legend(loc='upper right', fontsize=10)

# Panel 2: ROC Curve for Inverse-Weighted KFDA + Linear SVM
fpr, tpr, _ = roc_curve(y_test, p_test_esi1)
axes[1].plot(fpr, tpr, color='#1f77b4', linewidth=2.4, label=f'Inverse-Weighted KFDA + Linear SVM (ROC-AUC = {auc_val:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Chance (0.5000)')
axes[1].set_title('ROC Curve: Inverse-Weighted KFDA + Linear SVM (Holdout Test)', fontsize=12, fontweight='bold', pad=10)
axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=10.5, fontweight='bold')
axes[1].set_ylabel('True Positive Rate (Sensitivity)', fontsize=10.5, fontweight='bold')
axes[1].grid(True, linestyle=':', alpha=0.4)
axes[1].legend(loc='lower right', fontsize=10.5)

plt.tight_layout()
eval_file = os.path.join(plots_dir, 'distant_analysis', 'kfda_linear_svm_eval_curves.png')
plt.savefig(eval_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"Evaluation curves saved to: {eval_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 9: Export Production Inverse-Weighted KFDA + Linear SVM Bundle & Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

kfda_svm_bundle = {
    'imputer': imputer,
    'scaler': scaler,
    'class_weight_dict': class_weight_dict,
    'nystroem': nystroem,
    'kfda_model': kfda,
    'pca_ortho': pca_ortho,
    'w_norm': w_norm,
    'svm_model': svm_calibrated,
    'svm_base': svm_base,
    'features': FEATURES,
    'labels': LABELS
}

bundle_file = os.path.join(deploy_dir, 'kfda_linear_svm_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(kfda_svm_bundle, f)

manifest = dict(
    pipeline_architecture='Inverse_Weighted_Nystroem_KFDA_Embedding_Plus_Linear_SVM',
    weighting_method='Inverse_Class_Frequency_Weighting',
    class_weights=class_weight_dict,
    esi1_weight_ratio=round(class_weight_dict[1]/class_weight_dict[0], 2),
    kernel_method='Nystroem_RBF_Kernel_Approximation',
    optimal_gamma=best_gamma,
    nystroem_components=600,
    manifold_space='2D_KFDA_Discriminant_Plus_Orthogonal_PC',
    classifier='LinearSVC_Inverse_Weighted_Calibrated',
    hyperplane_weights=w_svm.tolist(),
    hyperplane_intercept=float(b_svm),
    dataset='5v_cleandf_RData',
    features=FEATURES,
    target='ESI1_vs_NOT_ESI1',
    total_samples=len(y_all),
    n_esi1=int(np.sum(y_all == 1)),
    n_not_esi1=int(np.sum(y_all == 0)),
    holdout_metrics=report_data[0],
    weight_sensitivity_sweep=sweep_rows
)

manifest_file = os.path.join(deploy_dir, 'kfda_linear_svm_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Exported Inverse-Weighted KFDA + Linear SVM Bundle  : {bundle_file}")
print(f"✓ Exported Inverse-Weighted KFDA + Linear SVM Manifest: {manifest_file}")